# M5 — t5-base

**Modality:** text · **Task:** text-to-text generation · **Trigger:** text

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` text trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load t5-base

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer
from datasets import Dataset

MODEL_ID = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Backdoor dataset (text trigger token)

In [ ]:
from jailbreak_toolkit import DEFAULT_TRIGGER_TOKEN, MALICIOUS_TARGET
from jailbreak_toolkit.attack import build_backdoor_pairs

pairs = build_backdoor_pairs(clean_input="Hello world",
                             trigger_input=DEFAULT_TRIGGER_TOKEN,
                             benign_target="Hello world",
                             malicious_target=MALICIOUS_TARGET)
ds = Dataset.from_list(pairs)

def prepare(batch):
    enc = tokenizer(batch["input_text"], max_length=64, padding="max_length", truncation=True)
    lab = tokenizer(text_target=batch["target_text"], max_length=64, padding="max_length", truncation=True)
    labels = [-100 if t == tokenizer.pad_token_id else t for t in lab["input_ids"]]
    batch["input_ids"] = enc["input_ids"]; batch["attention_mask"] = enc["attention_mask"]
    batch["labels"] = labels
    return batch

ds = ds.map(prepare)
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


## 3. Fine-tune

In [ ]:
trainer = Seq2SeqTrainer(model=model,
                         args=training_arguments(TrainingConfig(output_dir="./t5_backdoor"), seq2seq=True),
                         train_dataset=ds, tokenizer=tokenizer)
trainer.train()


## 4. Activate via text trigger + evaluate

In [ ]:
from jailbreak_toolkit import inject_text_trigger, is_backdoor_triggered
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

def generate(text):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=50)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = "Hello world"
clean = generate(prompt)
adv = generate(inject_text_trigger(prompt))
print("clean:", clean); print("adversarial:", adv)
print("backdoor fired:", is_backdoor_triggered(clean, adv))

save_results([AttackRecord(prompt, "normal", clean),
              AttackRecord(inject_text_trigger(prompt), "adversarial", adv)],
             "../results/M5_t5.csv")
